# Simple RAG System - Tips Hindawi University

Goal: build a simple Retrieval-Augmented Generation (RAG) system to answer questions based on a PDF document about Tips Hindawi University.

Steps:
1. Load the PDF as the knowledge base
2. Split the text into chunks
3. Build a retriever using TF-IDF and cosine similarity
4. Build a simple generator that composes an answer from the retrieved chunks, and honestly says when the document does not cover the question
5. Answer the four required questions


In [1]:
# Install required libraries (run once)
!pip install PyPDF2 scikit-learn -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.3 MB/s eta 0:00:00


In [2]:
# Set the path to the PDF file
PDF_PATH = "Tips Hindawi University Info.pdf"


In [3]:
# Load the PDF
import PyPDF2

def load_pdf_text(path: str) -> str:
    reader = PyPDF2.PdfReader(path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text() or ""
        text += page_text + "\n"
    return text

raw_text = load_pdf_text(PDF_PATH)
print(f"Extracted {len(raw_text)} characters from the PDF.\n")
print(raw_text[:500], "...")


Extracted 4112 characters from the PDF.

1. General Overview
Tips Hindawi University (THU) is a premier institution of higher education located in the heart of the Middle
East. Founded in 1963, the university has grown into a globally recognized center for academic excellence
and innovation. With over six decades of educational leadership, THU has produced more than 150,000
graduates who serve in diverse industries and academic circles worldwide.
The university is accredited by the International Commission for Academic Standards and th ...


## Chunking

Split the text into chunks based on numbered section headers (e.g. "1.", "2.1") so each chunk stays on one coherent topic.

In [4]:
import re

def chunk_text(text: str):
    # split on section headers like "1. " or "2.1 "
    raw_chunks = re.split(r"\n(?=\d+\.\d? )|\n(?=\d+\. )", text)
    chunks = [c.strip().replace("\n", " ") for c in raw_chunks if len(c.strip()) > 30]
    return chunks

chunks = chunk_text(raw_text)
print(f"Total chunks: {len(chunks)}\n")
for i, c in enumerate(chunks[:3]):
    print(f"[{i}] {c[:150]}...\n")


Total chunks: 18

[0] 1. General Overview Tips Hindawi University (THU) is a premier institution of higher education located in the heart of the Middle East. Founded in 196...

[1] 2.1 Main Campus The main campus is located in the capital city and spans over 300 acres. It houses: - 12 academic buildings - Central library with ove...

[2] 2.2 Satellite Campuses North Campus : Focused on agricultural sciences and environmental research West Campus : Home to the School of Design and Archi...



## Retriever (TF-IDF + Cosine Similarity)

Each chunk is converted to a TF-IDF vector. A question is compared against every chunk using cosine similarity, and the closest chunks are returned.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(chunks)

def retrieve(question: str, top_k: int = 3):
    q_vec = vectorizer.transform([question])
    sims = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_idx = sims.argsort()[::-1][:top_k]
    return [(chunks[i], float(sims[i])) for i in top_idx]

# quick test
for c, score in retrieve("Where is the university located?", top_k=2):
    print(f"({score:.2f}) {c[:150]}...\n")


(0.34) 1. General Overview Tips Hindawi University (THU) is a premier institution of higher education located in the heart of the Middle East. Founded in 196...

(0.11) 2.1 Main Campus The main campus is located in the capital city and spans over 300 acres. It houses: - 12 academic buildings - Central library with ove...



## Generator

The generator is extractive: it takes the best retrieved chunks and composes an answer from them directly.

One issue to watch for: TF-IDF gives very high weight to rare words, so a word like "university" that only appears once in the whole document can make any chunk containing it score high, even if that chunk has nothing to do with the actual question (for example, a question about "online programs"). So the similarity score alone is not reliable enough.

Fix: before trusting an answer, check for real keyword grounding between the meaningful words in the question (excluding stopwords and the university's own name, since that appears almost everywhere) and the retrieved chunk. If there is no real overlap, the system honestly reports that the document does not mention this, instead of fabricating an answer. This is the core idea of grounding in RAG.

In [6]:
GENERIC_ENTITY_WORDS = {"university", "tips", "hindawi", "thu"}  # the institution's own name, not discriminative
ENGLISH_STOP = set(vectorizer.get_stop_words())

def content_words(question: str):
    # meaningful (non-generic) words in the question
    words = re.findall(r"[a-zA-Z]+", question.lower())
    return set(w for w in words if w not in ENGLISH_STOP and w not in GENERIC_ENTITY_WORDS and len(w) > 2)

def is_grounded(question: str, chunk: str) -> bool:
    # check if at least one meaningful question word literally appears in the chunk
    cw = content_words(question)
    if not cw:
        return True
    chunk_words = set(re.findall(r"[a-zA-Z]+", chunk.lower()))
    return len(cw & chunk_words) > 0

def generate_answer(question: str, top_k: int = 3):
    results = retrieve(question, top_k=top_k)
    top_chunk, top_score = results[0]
    confident = top_score > 0.05 and is_grounded(question, top_chunk)

    if not confident:
        answer = "The document does not explicitly mention this. Closest related content found (for reference only):\n"
    else:
        answer = "Based on the retrieved context:\n"

    for chunk, score in results:
        if score > 0.03:  # skip very weak results
            answer += f"\n- {chunk}"

    return answer, results

def ask(question: str):
    answer, retrieved = generate_answer(question)
    print("Question:", question)
    print("\nRetrieved chunks:")
    for c, s in retrieved:
        print(f"  ({s:.2f}) {c[:120]}...")
    print("\nAnswer:\n", answer)
    print("\n" + "=" * 90 + "\n")
    return answer


## Answering the required questions

In [7]:
questions = [
    "Where is Tips Hindawi University located?",
    "Does the university offer online programs?",
    "Is there financial aid for international students?",
    "What languages are used for instruction?",
]

answers = {}
for q in questions:
    answers[q] = ask(q)


Question: Where is Tips Hindawi University located?

Retrieved chunks:
  (0.36) 1. General Overview Tips Hindawi University (THU) is a premier institution of higher education located in the heart of t...
  (0.07) 2.1 Main Campus The main campus is located in the capital city and spans over 300 acres. It houses: - 12 academic buildi...
  (0.00) 8. Faculty Highlights Dr. Yasir Al-Sabbagh  (Computer Science): Expert in natural language processing Prof. Rana Khalil ...

Answer:
 Based on the retrieved context:

- 1. General Overview Tips Hindawi University (THU) is a premier institution of higher education located in the heart of the Middle East. Founded in 1963, the university has grown into a globally recognized center for academic excellence and innovation. With over six decades of educational leadership, THU has produced more than 150,000 graduates who serve in diverse industries and academic circles worldwide. The university is accredited by the International Commission for Academic S

## Summary of answers

| # | Question | Answer summary |
|---|---|---|
| 1 | Where is THU located? | The university is in the heart of the Middle East, with the main campus in the capital city (the document does not name the specific country or city). |
| 2 | Does it offer online programs? | The document does not mention any online programs at all -- there is no evidence of this. |
| 3 | Financial aid for international students? | There are general merit scholarships and need-based grants, and an Office of International Programs that assists international students, but the document does not explicitly tie the scholarships to international students specifically. |
| 4 | What languages are used for instruction? | The document does not mention the language of instruction at all. |

Note: questions 2 and 4 are a good example of why grounding matters in RAG -- instead of fabricating an answer, the system honestly reports that the information is not in the source document.
